# 07 · Company comparison dashboard

**Research question:** How do the 20 companies compare across operating quality, valuation and historical price risk?

This dashboard combines the shared calculations used in the preceding notebooks. It is a comparison aid, not a score, rank or investment recommendation. The 2025 figures are provisional estimates known on 31 December 2025. Price returns are descriptive hindsight measures and exclude dividends.

## 1. Load approved data and calculate each component

Run from the repository root or `notebooks/`. No cleaning is repeated and no other notebook needs to have run first.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_cleaned_data
from src.research.fundamentals import latest_fundamentals
from src.research.price_risk import risk_summary
from src.research.valuation import valuation_snapshot
from src.research.presentation import style_extremes, METRIC_DIRECTIONS

annual, prices = load_cleaned_data(project_root / 'data' / 'Cleaned')
fundamentals = latest_fundamentals(annual)
valuation = valuation_snapshot(annual, prices)
price_risk = risk_summary(prices)
assert all(len(table) == 20 and table['ticker'].is_unique for table in [fundamentals, valuation, price_risk])
print('All three components include 20 unique tickers.')

## ⭐ 2. Consolidated comparison

Fractions in the calculation tables are converted to percentages only for display. Negative drawdown means a loss from a prior high; a value closer to zero is less severe. Industrial net debt/FCF is undefined when FCF is nonpositive and omitted for CDB, BSL and CRM. A low P/E alone cannot establish value.

In [ ]:
dashboard = (fundamentals[['ticker','company','sector','revenue_cagr_5y','profit_margin','fcf_margin','net_debt_to_fcf','loans_overdue','capital_cushion','claims_ratio','capital_coverage']]
    .merge(valuation[['ticker','pe_ratio','fcf_yield','dividend_yield','price_to_sales']], on='ticker', validate='one_to_one')
    .merge(price_risk[['ticker','cagr','annualised_volatility','max_drawdown','beta']], on='ticker', validate='one_to_one')
    .sort_values(['sector','ticker']).reset_index(drop=True))
assert len(dashboard) == 20
assert set(dashboard['ticker']) == set(annual['ticker'])
percent_columns = ['revenue_cagr_5y','profit_margin','fcf_margin','fcf_yield','dividend_yield','cagr','annualised_volatility','max_drawdown','loans_overdue','capital_cushion','claims_ratio','capital_coverage']
shown = dashboard.copy()
shown[percent_columns] = shown[percent_columns].mul(100)
shown = shown.rename(columns={c:c.replace('_',' ').title()+' (%)' for c in percent_columns})
print('2025 financials and valuation; full-history price risk (GBP, multiples and percentages as labelled)')
display(shown.round(2))

## 3. Category comparisons

Each category answers a different question. Read growth with cash generation, and valuation with business risk. Missing values remain visible rather than excluding a company.
**Units in the category tables:** fraction-valued growth, margins, yields, volatility, drawdown, overdue loans, capital cushions, claims ratios and capital cover use decimals (0.10 = 10%). P/E, net debt/FCF and beta are multiples.


In [ ]:
categories = {
    'Growth and profitability': ['ticker','company','sector','revenue_cagr_5y','profit_margin'],
    'Cash generation and industrial stability': ['ticker','company','sector','fcf_margin','net_debt_to_fcf'],
    'Valuation': ['ticker','company','sector','pe_ratio','fcf_yield','dividend_yield','price_to_sales'],
    'Historical price risk': ['ticker','company','sector','cagr','annualised_volatility','max_drawdown','beta'],
    'Specialist financial health': ['ticker','company','sector','loans_overdue','capital_cushion','claims_ratio','capital_coverage'],
}
for title, columns in categories.items():
    print(title)
    subset = dashboard[columns]
    if title == 'Specialist financial health':
        subset = subset.loc[subset[['loans_overdue','capital_cushion','claims_ratio','capital_coverage']].notna().any(axis=1)]
    display(subset.round(3))

## 4. Highlight valid extremes

For 20 valid observations, the top and bottom 10% are two companies each. For fewer valid observations, use the ceiling of 10%; fewer than ten valid values get no decile colour. Ties at the boundary are resolved alphabetically by ticker. Missing and infinite values are excluded. Green means favourable on the stated metric, red means adverse; blue and purple denote high and low **statistical** valuation extremes without calling either attractive. The signed drawdown measure is better when closer to zero.

In [ ]:
highlight_columns = ['ticker','company','sector','revenue_cagr_5y','profit_margin','fcf_margin','net_debt_to_fcf','pe_ratio','price_to_sales','cagr','annualised_volatility','max_drawdown']
print('Green = favourable; red = adverse; blue/purple = valuation extremes only')
display(style_extremes(dashboard[highlight_columns], METRIC_DIRECTIONS))

## 5. Sector-level comparisons

Each sector contains only four companies, so these tables use simple values rather than decile labels. Financials require separate measures for CDB and BSL (loans and capital) and CRM (claims and insurance cover). Cross-sector averages can conceal business-model differences.
**Units:** fraction-valued fields use decimals (0.10 = 10%); P/E and net debt/FCF are multiples.


In [ ]:
for sector, group in dashboard.groupby('sector', sort=True):
    print(f'{sector} — {len(group)} companies')
    if sector == 'Financials':
        cols = ['ticker','company','profit_margin','fcf_margin','pe_ratio','loans_overdue','capital_cushion','claims_ratio','capital_coverage']
    else:
        cols = ['ticker','company','revenue_cagr_5y','profit_margin','fcf_margin','net_debt_to_fcf','pe_ratio','annualised_volatility']
    display(group[cols].round(3))

## 6. Focused visual comparisons

Each point is labelled by ticker. Dashed zero lines distinguish cash generation from cash consumption. Scatter plots reveal tradeoffs but do not control for company size or sector differences.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plots = [
    ('revenue_cagr_5y','fcf_margin','Revenue CAGR 2020–2025 (%)','2025 FCF margin (%)','Growth and cash generation'),
    ('profit_margin','pe_ratio','2025 profit margin (%)','2025 P/E (x)','Profitability and valuation'),
    ('annualised_volatility','cagr','Full-history volatility (%)','Full-history price CAGR (%)','Historical risk and return'),
]
for ax,(x,y,xlabel,ylabel,title) in zip(axes, plots):
    subset = dashboard[['ticker',x,y]].dropna()
    xscale = 100 if x != 'pe_ratio' else 1
    yscale = 100 if y != 'pe_ratio' else 1
    ax.scatter(subset[x]*xscale, subset[y]*yscale, alpha=.75)
    for row in subset.itertuples(index=False):
        ax.annotate(row.ticker, (getattr(row,x)*xscale,getattr(row,y)*yscale), fontsize=7, xytext=(3,3), textcoords='offset points')
    ax.axhline(0, color='gray', linewidth=.7, linestyle='--')
    ax.set(title=title,xlabel=xlabel,ylabel=ylabel)
    ax.grid(alpha=.2)
fig.tight_layout()
plt.show()

## ⭐ 7. Observations and limits

- Revenue growth and FCF margin should be read together: growth with negative cash generation may require financing, while a positive margin does not prove future growth.
- **CDB** and **BSL** need overdue-loan and capital-cushion review; **CRM** needs claims and capital-cover review. Industrial leverage colours intentionally exclude them.
- Historical price return and volatility describe past experience, not a forecast. A high or low valuation multiple requires investigation of earnings quality and expectations.
- Follow-up: reconcile individual valuation outliers with factsheet descriptions, inspect the latest three to five years of FCF, and assess the fictional macro scenarios in Notebook 08.

**No final company ranking or allocation is made here.**